<a href="https://colab.research.google.com/github/peterhelfer/aleph/blob/main/code/aleph_pcn_cifar10_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Supplement to the manuscript "Introduction to Predictive Coding Networks for Machine Learning" by Monadillo (2025)*

*This notebook uses CUDA (GPU), but with minor modifications it could run on a CPU*

# Predictive Coding Network on CIFAR-10

This Python notebook implements a Predictive Coding Network (PCN) for supervised image classification on the CIFAR-10 dataset.

The notebook was written to run in Google Colab, as is.

In [78]:
import os
from google.colab import drive

DATASET_ROOT = '/content/drive/MyDrive/datasets'
DATA_ROOT = "/content/drive/MyDrive/data"

!fusermount -u /content/drive
!rm -rf /content/drive
!mkdir /content/drive
drive.mount('/content/drive', force_remount=True)
os.makedirs(DATASET_ROOT, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

"""
# Testing
foo = [1, 2, 3, 4, 5]
with open(f"{DATA_ROOT}/foo.pkl", "wb") as f:
        pickle.dump(foo, f)
"""

Mounted at /content/drive


'\n# Testing\nfoo = [1, 2, 3, 4, 5]\nwith open(f"{DATA_ROOT}/foo.pkl", "wb") as f:\n        pickle.dump(foo, f)\n'

In [79]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch.amp import autocast
import plotly.graph_objects as go
from plotly import colors
import numpy as np
import math

import sys, datetime, pickle



## Download and preprocess dataset, build dataloaders

In [80]:
BATCH_SIZE = 500

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

trainset = torchvision.datasets.CIFAR10(
    #root='./data',
    root=DATASET_ROOT,
    train=True,
    download=True,
    transform=transform
)

trainloader = DataLoader(
    trainset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=2
)

testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

testloader = DataLoader(
    testset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=2
)

print(f'Batch size: {BATCH_SIZE}, Train batches: {len(trainloader)}, Test batches: {len(testloader)}')

Batch size: 500, Train batches: 100, Test batches: 20


## Create split train and test sets

In [81]:
from torch.utils.data import Subset

sub_trainsets = []
sub_testsets = []
sub_trainloaders = []
sub_testloaders = []

"""
groups = [
    [0]
]
"""

"""
groups = [
    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
]
"""

groups = [
    [0, 1],     # airplane, car
    [2, 3],     # bird, cat
    [4, 5],     # deer, dog
    [6, 7],     # frog, horse
    [8, 9]      # ship, truck
]

for i, class_ids in enumerate(groups):
    print(i, class_ids)

    train_indices = [j for j, t in enumerate(trainset.targets) if t in class_ids]
    sub_trainset = Subset(trainset, train_indices)
    sub_trainsets.append(sub_trainset)

    sub_trainloaders.append(
        DataLoader(
          sub_trainset,
          batch_size=BATCH_SIZE,
          shuffle=True,
          num_workers=2,
          pin_memory=True,
          prefetch_factor=2)
    )

    test_indices = [j for j, t in enumerate(testset.targets) if t in class_ids]
    sub_testset = Subset(testset, test_indices)
    sub_testsets.append(sub_testset)

    sub_testloaders.append(
        DataLoader(
          sub_testset,
          batch_size=BATCH_SIZE,
          shuffle=True,
          num_workers=2,
          pin_memory=True,
          prefetch_factor=2)
    )


for tr, te in zip(sub_trainsets, sub_testsets):
    print(f'Batch size: {BATCH_SIZE}, Train batches: {len(tr.indices)}, Test batches: {len(te.indices)}')
print('-----')

#print('Goodbye!')
#sys.exit(0)


0 [0, 1]
1 [2, 3]
2 [4, 5]
3 [6, 7]
4 [8, 9]
Batch size: 500, Train batches: 10000, Test batches: 2000
Batch size: 500, Train batches: 10000, Test batches: 2000
Batch size: 500, Train batches: 10000, Test batches: 2000
Batch size: 500, Train batches: 10000, Test batches: 2000
Batch size: 500, Train batches: 10000, Test batches: 2000
-----


## Predictive Coding Network Components

In [82]:
# Define latent layer
class PCNLayer(nn.Module):
    def __init__(self,
                 in_dim,   # d_{l+1}  - dimension of layer above
                 out_dim,  # d_l      - dimension of current layer
                 activation_fn=torch.relu, # nonlinearity f^(l)
                 activation_deriv=lambda a: (a > 0).float() # derivative f^(l)'
                 ):
        super().__init__()
        self.W = nn.Parameter(torch.empty(out_dim, in_dim)) # W^(l)
        nn.init.xavier_uniform_(self.W)
        self.activation_fn     = activation_fn
        self.activation_deriv  = activation_deriv

    def forward(self, x_above):
        with autocast(device_type='cuda'):
            a     = x_above @ self.W.T      # A^(l) = X^(l+1) @ {W^(l)}^T
            x_hat = self.activation_fn(a)   # \hat X^(l) = f^(l)(A^(l))
            return x_hat, a


# Define network structure
class PredictiveCodingNetwork(nn.Module):
    def __init__(self,
                 dims,        # [d_0,...,d_L]  - list of layer dimensions
                 output_dim   # d_out          - readout layer dimension
                 ):
        super().__init__()
        self.dims = dims
        self.L = len(dims) - 1            # L  - number of latent layers
        self.layers = nn.ModuleList([     # Build latent layers
            PCNLayer(in_dim=dims[l+1],    # Layer l reads from layer l+1
                     out_dim=dims[l])
            for l in range(self.L)        # l = 0,...,L-1
        ])
        # Build readout layer: maps top latent X^(L) to
        # predicted output \hat Y
        # Note: nn.Linear applies (batch, in_features) @ weight.T under the hood,
        # which corresponds exactly to X^(L) @ (W^out)^T
        self.readout = nn.Linear(dims[-1], output_dim, bias=False)


    def init_latents(self, batch_size, device):
        # returns [X^(1),...,X^(L)] as random normals
        return [
           torch.randn(batch_size, d, device=device, requires_grad=False)
           for d in self.dims[1:]
        ]

    def compute_errors(self, inputs_latents):
        # Compute predictions from input and latent variables
        # Argument: inputs_latents - list of tensors [X^(0), X^(1),...,X^(L)] shaped [(B,d_0),...,(B,d_L)]
        # Returns: two lists of tensors shaped [(B,d_0),...,(B,d_{L-1})]
        errors, gain_modulated_errors = [], []
        for l, layer in enumerate(self.layers):       # l = 0,...,L-1
            # Call to layer returns:
            #   a = X^(l+1) @ W^(l).T  (preactivations A^(l))
            #   x_hat = f^(l)(a)       (predictions \hat X^(l))
            x_hat, a  = layer(inputs_latents[l + 1])
            err       = inputs_latents[l] - x_hat
            gm_err    = err * layer.activation_deriv(a)
            errors.append(err)                               # E^(l) - prediction errors
            gain_modulated_errors.append(gm_err)             # H^(l) - gain-modulated errors
        return errors, gain_modulated_errors

# Globals





In [83]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

energy_history = []             # Will hold per-epoch batch-averaged energy trajectories
supervised_energy_history = []  # Will hold per-epoch batch-averaged supervised energy trajectories

Using device: cuda


# Utilities

In [84]:
class RunningStats:
    "Compute running mean and stdev for a series of float values"
    def __init__(self):
        self.n = 0
        self.m = 0
        self.new_m = 0
        self.s = 0
        self.new_s = 0
        self.mean_ = 0
        self.variance_ = 0
        self.stdev_ = 0
        self.is_dirty = True

    def clear(self):
        self.n = 0

    def push(self, x):
        self.n += 1

        if self.n == 1:
            self.m = self.new_m = x
            self.s = 0
        else:
            self.new_m = self.m + (x - self.m) / self.n
            self.new_s = self.s + (x - self.m) * (x - self.new_m)

            self.m = self.new_m
            self.s = self.new_s
        self.is_dirty = True

    def update(self):
        self.mean_ = self.new_m if self.n else 0.0
        self.variance_ = self.new_s / (self.n - 1) if self.n > 1 else 0.0
        self.stdev_ = math.sqrt(self.variance_)
        self.is_dirty = False

    def mean(self):
        if self.is_dirty: self.update()
        return self.mean_

    def variance(self):
        if self.is_dirty: self.update()
        return self.variance_

    def stdev(self):
        if self.is_dirty: self.update()
        return self.stdev_

# Timestamp functions
#
def datetime_str(dt, dsep='_', dtsep='__', tsep='_'):
    """Build a datetime string like yyyy_mm_dd__hh_mm_ss
    :param dt: a datetime.datetime object
    :param dsep: separator between year, month and day
    :param dtsep: separator between  date and time parts
    :param tsep: separator between hour, minutes and seconds"""
    return (f'{dt.year}{dsep}{dt.month:02d}{dsep}{dt.day:02d}{dtsep}'
            f'{dt.hour:02d}{tsep}{dt.minute:02d}{tsep}{dt.second:02d}')

#def now_str(dsep='_', dtsep='__', tsep='_'):
def now_str(**kwargs):
    """Build a string representation of current date and time"""
    return datetime_str(datetime.datetime.now(), **kwargs)

"""
# Testing

rs = [[RunningStats() for _ in range(5)] for _ in range(5)]

with open(f"{DATA_ROOT}/rs_{now_str()}.pkl", "wb") as f:
    pickle.dump(rs, f)
"""

'\n# Testing\n\nrs = [[RunningStats() for _ in range(5)] for _ in range(5)]\n\nwith open(f"{DATA_ROOT}/rs_{now_str()}.pkl", "wb") as f:\n    pickle.dump(rs, f)\n'

## Update Rules (Inference and Learning)

## Training Loop

In [85]:
def train_pcn(model, data_loader, num_epochs, eta_infer, eta_learn, T_infer, T_learn, device='cuda'):
    model.to(device)
    model.train()

    global energy_history
    global supervised_energy_history

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1} / {num_epochs}")

        epoch_energies = []             # Will hold batch-averaged energy trajectories for this epoch
        epoch_supervised_energies = []  # Will hold batch-averaged supervised energy trajectories for this epoch

        #for x_batch, y_batch in tqdm(data_loader):
        for x_batch, y_batch in tqdm(data_loader):
            B = x_batch.size(0)
            d_0 = model.dims[0]
            batch_energies = []            # Will hold the batch-averaged energy trajectory for this batch
            batch_supervised_energies = [] # Will hold the batch-averaged supervised energy trajectory for this batch

            # Flatten inputs
            x_batch = x_batch.view(B, d_0).to(device)
            # Convert ouputs to one-hot vectors
            y_batch = F.one_hot(y_batch, num_classes=model.readout.out_features).float().to(device)
            # Concatenate inputs X^(0) and initialized latents X^(l) - shape [(B,d_0),...,(B,d_L)]
            inputs_latents = [x_batch] + model.init_latents(B, device)
            # Weights W^(0),...,W^(L-1),W^out of latent and output layers - shape [(d_0,d_1),...,(d_{L-1},d_L),(d_L,d_out)]
            weights = [layer.W for layer in model.layers] + [model.readout.weight]


            # Prediction block - for energy at step t=0 and inference at step t=1
            # Compute E^(l) and H^(l) for l=0,...,L-1
            errors, gain_modulated_errors = model.compute_errors(inputs_latents)
            # Compute \hat Y from X^(L)
            y_hat           = model.readout(inputs_latents[-1]) # X^(L) @ W^out.T
            # Compute E^sup
            eps_sup         = y_hat - y_batch
            # Compute E^(L)
            eps_L           = eps_sup @ weights[-1]
            # Append to E^(L) to errors [E^(0),...,E^(L-1)]
            errors_extended = errors + [eps_L]


            # Record initial batch-averaged energy before any updates (t=0)
            supervised_energy = 0.5 * eps_sup.pow(2).sum().item() / B
            latent_energy     = 0.5 * sum(e.pow(2).sum().item() for e in errors) / B
            batch_supervised_energies.append(supervised_energy)
            batch_energies.append(latent_energy + supervised_energy)

            # INFERENCE - T_infer steps
            with torch.no_grad(), autocast(device_type='cuda'):
                for t in range(1, T_infer + 1):

                    # Predictions for this inference step t have already been computed
                    # Inference updates - Gradient step for latents X^(1),...,X^(L)
                    for l in range(1, model.L + 1):  # l=1,...,L
                        grad_Xl = errors_extended[l] - gain_modulated_errors[l-1] @ weights[l-1]
                        inputs_latents[l] -= eta_infer * grad_Xl


                    # Prediction block - for energies at the end of this step t, and for inference/learning at step t+1
                    errors, gain_modulated_errors = model.compute_errors(inputs_latents)
                    y_hat           = model.readout(inputs_latents[-1]) # X^(L) @ W^out.T
                    eps_sup         = y_hat - y_batch
                    eps_L           = eps_sup @ weights[-1]
                    errors_extended = errors + [eps_L]


                    # Record batch-averaged energy at this inference step t
                    supervised_energy = 0.5 * eps_sup.pow(2).sum().item() / B
                    latent_energy     = 0.5 * sum(e.pow(2).sum().item() for e in errors) / B
                    batch_supervised_energies.append(supervised_energy)
                    batch_energies.append(latent_energy + supervised_energy)

            # LEARNING - T_learn steps
            with torch.no_grad():           # no autocast - keep precision in weight updates
                for t in range(T_infer + 1, T_learn + T_infer + 1):

                    # Predictions for this learning step t have already been computed
                    # Gradient step for W^(0),...,W^(l-1)
                    for l in range(model.L): # l=0,...,L-1
                        grad_Wl = -(gain_modulated_errors[l].T @ inputs_latents[l+1]) / B
                        weights[l] -= eta_learn * grad_Wl
                    # Gradient step for W^out
                    grad_Wout = eps_sup.T @ inputs_latents[-1] / B
                    weights[-1] -= eta_learn * grad_Wout


                    # Prediction block - for energies at the end of this step t, and for learning at step t+1
                    errors, gain_modulated_errors = model.compute_errors(inputs_latents)
                    y_hat           = model.readout(inputs_latents[-1]) # X^(L) @ W^out.T
                    eps_sup         = y_hat - y_batch


                    # Record batch-averaged energy at this learning step t
                    supervised_energy = 0.5 * eps_sup.pow(2).sum().item() / B
                    latent_energy     = 0.5 * sum(e.pow(2).sum().item() for e in errors) / B
                    batch_supervised_energies.append(supervised_energy)
                    batch_energies.append(latent_energy + supervised_energy)



            # Save this batch’s trajectory to this epoch's list
            epoch_energies.append(batch_energies)
            epoch_supervised_energies.append(batch_supervised_energies)

        # Save this epoch's list of all batch trajectories
        energy_history.append(epoch_energies)
        supervised_energy_history.append(epoch_supervised_energies)

    return energy_history, supervised_energy_history

# Test loop

In [86]:
@torch.no_grad()
def test_pcn(model, testloader, T_infer, eta_infer, device='cuda'):
    """
    Test a PCN: for each batch, run T_infer inference steps
    updating X^(1),...,X^(L)) exactly as in training, then
    compute Top-1 and Top-3 accuracy on the final readout.
    """
    model.to(device).eval()
    total, top1_correct, top3_correct = 0, 0, 0

    #for x_batch, y_batch in tqdm(testloader):
    for x_batch, y_batch in testloader:
        B = x_batch.size(0)
        total += B
        d_0 = model.dims[0]
        # Flatten inputs
        x_batch = x_batch.view(B, d_0).to(device)
        # Preserve integer labels for metrics
        y_labels = y_batch.to(device)
        # Convert outputs to one-hot vectors for inference
        y_batch = F.one_hot(y_labels, num_classes=model.readout.out_features) \
                       .float().to(device)

        # Concatenate inputs X^(0) and initialized latents X^(l)
        inputs_latents = [x_batch] + model.init_latents(B, device)
        # Weights W^(0),...,W^(L-1),W^out
        weights = [layer.W for layer in model.layers] + [model.readout.weight]

        # INFERENCE - T_infer steps
        with autocast(device_type='cuda'):
            for t in range(1, T_infer + 1):
                errors, gain_modulated_errors = model.compute_errors(inputs_latents)
                y_hat           = model.readout(inputs_latents[-1])
                eps_sup         = y_hat - y_batch
                eps_L           = eps_sup @ weights[-1]
                errors_extended = errors + [eps_L]

                for l in range(1, model.L + 1):
                    grad_Xl = errors_extended[l] - gain_modulated_errors[l-1] @ weights[l-1]
                    inputs_latents[l] -= eta_infer * grad_Xl

        # METRICS
        logits = model.readout(inputs_latents[-1])       # (B, d_out)
        # Top-1
        preds1 = logits.argmax(dim=1)
        top1_correct += (preds1 == y_labels).sum().item()
        # Top-3
        _, preds3 = logits.topk(3, dim=1)                # (B, 3)
        top3_correct += (preds3 == y_labels.unsqueeze(1)).any(dim=1).sum().item()

    acc1 = top1_correct / total
    acc3 = top3_correct / total
    return acc1, acc3

# Plotting tools

In [87]:
def plot_energy_history_interactive(energy_history, T_infer, T_learn,
                                    title='Batch-averaged Energy Trajectories'
                                    ):
    """
    Interactive energy trajectories with Plotly.
    - energy_history: list of epochs; each epoch is list of batch energy lists.
    - T_infer: number of inference steps.
    - T_learn: number of learning steps.
    Hover to see Epoch, Batch, Step, Energy.
    """
    num_epochs = len(energy_history)
    # Sample one color per epoch from Viridis
    epoch_colors = colors.sample_colorscale(
        colors.sequential.Viridis,
        [i/(num_epochs-1) if num_epochs>1 else 0 for i in range(num_epochs)]
    )

    fig = go.Figure()
    for epoch_idx, epoch_energies in enumerate(energy_history):
        color = epoch_colors[epoch_idx]
        for batch_idx, batch_vals in enumerate(epoch_energies):
            steps = list(range(len(batch_vals)))
            # Attach epoch and batch metadata for hover
            customdata = [[epoch_idx+1, batch_idx+1] for _ in steps]
            fig.add_trace(go.Scatter(
                x=steps,
                y=batch_vals,
                mode='lines',
                line=dict(color=color, width=1),
                hovertemplate=(
                    'Epoch %{customdata[0]}<br>'
                    'Batch %{customdata[1]}<br>'
                    'Step %{x}<br>'
                    'Energy %{y:.4f}<extra></extra>'
                ),
                customdata=customdata,
                showlegend=False
            ))
    # Vertical separator between inference and learning
    sep_position = T_infer + 0.5
    fig.add_vline(
        x=sep_position,
        line_dash='dash',
        line_color='black',
        annotation_text='Inference end',
        annotation_position='top right'
    )

    fig.update_layout(
        title=title,
        xaxis_title='Step t (Inference then Learning)',
        yaxis_title='Energy'
    )

    # Optional: log-scale y-axis if needed
    # fig.update_yaxes(type='log')

    fig.show()

In [88]:
def plot_epoch_avg_interactive(energy_history, T_infer, T_learn,
                               title='Batch-Averaged Energy Trajectories (Mean ± 1-std)'
                              ):
    """
    Interactive per-epoch mean ±1-std energy trajectories using Plotly.
    - energy_history: list of epochs; each epoch is list of batch energy trajectories.
    - T_infer: number of inference steps.
    - T_learn: number of learning steps.
    Hover to see Epoch, Step, Mean Energy.
    """
    num_epochs = len(energy_history)
    epoch_colors = colors.sample_colorscale(
        colors.sequential.Viridis,
        [i/(num_epochs-1) if num_epochs>1 else 0 for i in range(num_epochs)],
        colortype='hex'   # Makes each color something like '#440154'
    )

    epoch_colors = [
        c if isinstance(c, str)
          else '#{0:02x}{1:02x}{2:02x}'.format(
              *(int(round(255*v)) for v in c)
          )
        for c in epoch_colors
    ]

    fig = go.Figure()
    total_steps = T_infer + T_learn

    for epoch_idx, epoch_batches in enumerate(energy_history):
        arr = np.array(epoch_batches)  # shape (n_batches, total_steps+1)
        mean = arr.mean(axis=0)
        std = arr.std(axis=0)
        steps = list(range(len(mean)))
        color = epoch_colors[epoch_idx]
        # Upper bound (mean + std)
        fig.add_trace(go.Scatter(
            x=steps,
            y=mean + std,
            mode='lines',
            line=dict(width=0),
            showlegend=False,
            hoverinfo='skip'
        ))

        # Lower bound (mean - std) with fill
        fig.add_trace(go.Scatter(
            x=steps,
            y=mean - std,
            mode='lines',
            fill='tonexty',
            # Convert hex '#rrggbb' → rgba(r,g,b,0.2)
            fillcolor=f"rgba({int(color[1:3],16)},"
                      f"{int(color[3:5],16)},"
                      f"{int(color[5:7],16)},0.2)",
            line=dict(width=0),
            showlegend=False,
            hoverinfo='skip'
        ))

        # Mean line
        fig.add_trace(go.Scatter(
            x=steps,
            y=mean,
            mode='lines',
            line=dict(color=color, width=2),
            name=f'Epoch {epoch_idx+1}',
            hovertemplate=(
                f'Epoch {epoch_idx+1}<br>'
                'Step %{x}<br>'
                'Energy %{y:.4f}<extra></extra>'
            )
        ))


    # Vertical separator between inference and learning
    fig.add_vline(
        x=T_infer + 0.5,
        line_dash='dash',
        line_color='black',
        annotation_text='Inference end',
        annotation_position='top right'
    )

    fig.update_layout(
        title=title,
        xaxis_title='Step t (Inference then Learning)',
        yaxis_title='Energy'
    )

    # Optional: log-scale y-axis if needed
    # fig.update_yaxes(type='log')

    fig.show()

# Model Instantiation and Training Execution

In [89]:
layer_dims = [3072, 1000, 500, 10]
output_dim = 10
pcn_model = PredictiveCodingNetwork(dims=layer_dims, output_dim=output_dim)

Hyperparameters for training

In [90]:
NUM_EPOCHS = 1 #4
ETA_INFER = 0.05
ETA_LEARN = 0.005
T_INFER = 50
T_LEARN = BATCH_SIZE # = 500 in this experiment

Train model, store energies for plotting

In [91]:
def train_pcn_subset(index):
    global device

    print(f"Training subset {index:d}...")
    energy_history, supervised_energy_history = train_pcn(
        model=pcn_model,
        #data_loader=trainloader,
        data_loader=sub_trainloaders[index], #PH
        num_epochs=NUM_EPOCHS,
        eta_infer=ETA_INFER,
        eta_learn=ETA_LEARN,
        T_infer=T_INFER,
        T_learn=T_LEARN,
        device=device
    )

    #print("\nTraining finished.")

Report Test Accuracies

In [92]:
def test_pcn_subset(index):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    acc1, acc3 = test_pcn(
        model=pcn_model,
        #testloader=testloader,
        testloader=sub_testloaders[index], #PH
        T_infer=T_INFER,
        eta_infer=ETA_INFER,
        device=device
    )
    #print("")
    print(f"    Test Accuracy subset {index:d} Top-1: {acc1*100:.2f}%  Top-3: {acc3*100:.2f}%")
    return acc1, acc3

In [93]:
# Accumulate mean accuracy and stdev in an array of RunningStats objects
# where rs[i][j] holds the values for testing subset j after training subset i.

#rng = np.random.default_rng()

num_subsets = len(sub_trainloaders)

NUM_RUNS = 10

for run in range(NUM_RUNS):
    print(f"Run {run+1} / {NUM_RUNS}")

    rs = [[RunningStats() for _ in range(num_subsets)] for _ in range(num_subsets)]

    for i in range(num_subsets):
        train_pcn_subset(i)
        for j in range(0, i+1):
            acc1, acc3 = test_pcn_subset(j)
            rs[i][j].push(acc1)

    for i in range(num_subsets):
        for j in range(0, i+1):
            m = rs[i][j].mean()
            s = rs[i][j].stdev()
            print(f"[{i},{j}]  Mean: {m: .2f}  Stdev: {s: .2f}")

    # Save rs to drive under a unique name, so we don't have to recompute it
    # if the runtime is disconnected.
    with open(f"{DATA_ROOT}/rs_{now_str()}.pkl", "wb") as f:
        pickle.dump(rs, f)

    # Also save it as "latest_rs"
    with open(f"{DATA_ROOT}/rs_latest.pkl", "wb") as f:
        pickle.dump(rs, f)


Run 1 / 10
Training subset 0...
Epoch 1 / 1


100%|██████████| 20/20 [00:27<00:00,  1.37s/it]


    Test Accuracy subset 0 Top-1: 100.00%  Top-3: 100.00%
Training subset 1...
Epoch 1 / 1


100%|██████████| 20/20 [00:26<00:00,  1.32s/it]


    Test Accuracy subset 0 Top-1: 56.05%  Top-3: 98.10%
    Test Accuracy subset 1 Top-1: 100.00%  Top-3: 100.00%
Training subset 2...
Epoch 1 / 1


...

    Test Accuracy subset 0 Top-1: 99.20%  Top-3: 100.00%
    Test Accuracy subset 1 Top-1: 98.85%  Top-3: 100.00%
    Test Accuracy subset 2 Top-1: 100.00%  Top-3: 100.00%
Training subset 3...
Epoch 1 / 1


100%|██████████| 20/20 [00:25<00:00,  1.29s/it]


    Test Accuracy subset 0 Top-1: 98.05%  Top-3: 100.00%
    Test Accuracy subset 1 Top-1: 99.75%  Top-3: 100.00%
    Test Accuracy subset 2 Top-1: 99.90%  Top-3: 100.00%
    Test Accuracy subset 3 Top-1: 100.00%  Top-3: 100.00%
Training subset 4...
Epoch 1 / 1


100%|██████████| 20/20 [00:25<00:00,  1.30s/it]


    Test Accuracy subset 0 Top-1: 99.95%  Top-3: 100.00%
    Test Accuracy subset 1 Top-1: 99.20%  Top-3: 100.00%
    Test Accuracy subset 2 Top-1: 99.85%  Top-3: 100.00%
    Test Accuracy subset 3 Top-1: 100.00%  Top-3: 100.00%
    Test Accuracy subset 4 Top-1: 100.00%  Top-3: 100.00%
[0,0]  Mean:  1.00  Stdev:  0.00
[1,0]  Mean:  1.00  Stdev:  0.00
[1,1]  Mean:  1.00  Stdev:  0.00
[2,0]  Mean:  0.99  Stdev:  0.00
[2,1]  Mean:  0.99  Stdev:  0.00
[2,2]  Mean:  1.00  Stdev:  0.00
[3,0]  Mean:  0.98  Stdev:  0.00
[3,1]  Mean:  1.00  Stdev:  0.00
[3,2]  Mean:  1.00  Stdev:  0.00
[3,3]  Mean:  1.00  Stdev:  0.00
[4,0]  Mean:  1.00  Stdev:  0.00
[4,1]  Mean:  0.99  Stdev:  0.00
[4,2]  Mean:  1.00  Stdev:  0.00
[4,3]  Mean:  1.00  Stdev:  0.00
[4,4]  Mean:  1.00  Stdev:  0.00


In [94]:
for i in range(num_subsets):
    for j in range(0, i+1):
        m = rs[i][j].mean()
        s = rs[i][j].stdev()
        print(rs[j][j].mean())

1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0


Plot batch-averaged energy trajectories

In [ ]:
"""
plot_energy_history_interactive(energy_history, T_infer, T_learn)

plot_energy_history_interactive(
    supervised_energy_history,
    T_infer, T_learn,
    title='Batch-Averaged Supervised Energy Trajectories'
)
"""

Plot epoch-averaged energy trajectories

In [ ]:
"""
plot_epoch_avg_interactive(energy_history, T_infer, T_learn)

plot_epoch_avg_interactive(
    supervised_energy_history,
    T_infer, T_learn,
    title='Batch-Averaged Supervised Energy Trajectories (Mean ± 1-std)'
)
"""

Save model weights **LOCALLY in Colab** - Make sure to store them elsewhere if needed later

(You can for example download the file to your computer from the *Files* tab before the Colab session ends)

In [ ]:
# Save the model
pcn_model.to('cpu') # Move model to CPU for compatibility

# Define the path to save the weights
save_path = 'pcn_model_statedict.pth'

# Save the model's state dictionary
torch.save(pcn_model.state_dict(), save_path)

print(f"Model state_dict saved to {save_path}")

In [ ]:
# How to load saved weights back into an identical model structure:

# Step 1: Recreate the model with the same architecture
# model_reloaded = PredictiveCodingNetwork(dims=layer_dims, output_dim=output_dim)

# OPTION 1 - load the weights from the GitHub repo mentioned in the manuscript
# from torch.hub import load_state_dict_from_url
# url = 'https://github.com/Monadillo/pcn-intro/raw/374774c6c493d419f266ab4fc305228a0dedae0b/pcn_model_statedict.pth'
# state_dict = load_state_dict_from_url(url, map_location='cpu')
# model_reloaded.load_state_dict(state_dict)

# OPTION 2 - load weights from a file you have saved locally
# save_path = 'pcn_model_statedict.pth' # or similar
# state_dict = torch.load(save_path, map_location='cpu')
# model_reloaded.load_state_dict(state_dict)

# Step 2: Move model to GPU if available
# if torch.cuda.is_available():
#    model_reloaded.to('cuda')

# Step 3: Set to evaluation mode (if the intention is to do inference only)
# model_reloaded.eval()